# Scenario: The "Silent" Vital Signs

In [3]:
import pandas as pd
import sqlite3
import numpy as np
# creating a dataset with missing null valuse
triage_data = {
  "triage_id": [5001, 5002, 5003, 5004, 5005],
  "patient_name": ["Alice Wong", "Bob Miller", "Charlie Davis", "Diana Prince", "Edward Norton"],
  "heart_rate": [72, None, 85, None, 60],  # None becomes null in sql
  "resp_rate": [16, 18, None, None, 14],
  "ai_urgency_score": [3, 5, 5, 5, 2] # Notice the 5s (Non-urgent) for missing data!
}
# adding dataset to DataFrame
df_triage = pd.DataFrame(triage_data)
# creating qsl connection
connt = sqlite3.connect(":memory:")
df_triage.to_sql("triage_audit", connt, index = False, if_exists="replace")
#function to run the query
def run_query(query):
  return pd.read_sql_query(query,connt)
print("++++++++++++++++++++++++++++++ Dayd 12 null value audit database is ready ++++++++++++++++++++ ")
print()

++++++++++++++++++++++++++++++ Dayd 12 null value audit database is ready ++++++++++++++++++++ 



# Finding the "Silent" Row

In [5]:
# query for all data to view
all_data = "SELECT * FROM triage_audit"
print("+++++++++++++++++++++++ all data to view +++++++++++++++++")
display(run_query(all_data))
print()
# query to find the patient_name for all patients where either the heart_rate OR the resp_rate is NULL.
null_values = """
SELECT * FROM triage_audit
WHERE heart_rate IS NULL OR resp_rate IS NULL
"""
print("+++++++++++++++++++++++++++++ paients with null value  +++++++++++++++++++++")
display(run_query(null_values))
print()

+++++++++++++++++++++++ all data to view +++++++++++++++++


,triage_id,patient_name,heart_rate,resp_rate,ai_urgency_score
0,5001,Alice Wong,72.0,16.0,3
1,5002,Bob Miller,NaN,18.0,5
2,5003,Charlie Davis,85.0,NaN,5
3,5004,Diana Prince,NaN,NaN,5
4,5005,Edward Norton,60.0,14.0,2



+++++++++++++++++++++++++++++ null value sections +++++++++++++++++++++


,triage_id,patient_name,heart_rate,resp_rate,ai_urgency_score
0,5002,Bob Miller,NaN,18.0,5
1,5003,Charlie Davis,85.0,NaN,5
2,5004,Diana Prince,NaN,NaN,5


# Impact Summary

In [12]:
# query that counts how many patients have missing data, grouped by the ai_urgency_score.
score5_patients = """
SELECT ai_urgency_score, COUNT(*) FROM triage_audit
GROUP BY ai_urgency_score
"""
print("+++++++++++++++++++++++++++++ ai_urgency_score Reslults +++++++++++++++++++")
display(run_query(score5_patients))


+++++++++++++++++++++++++++++ ai_urgency_score Reslults +++++++++++++++++++


,ai_urgency_score,COUNT(*)
0,2,1
1,3,1
2,5,3
